# Data Exploration

Explore the historical game data available for hype score backtesting.

Current rating snapshots are not historical pregame evidence. The scoring functions leave qualification unavailable for past or started games. Do not interpret these comparisons as validation of historical playoff stakes, or use them to tune those stakes. Strength and championship inputs also lack a historical evidence guarantee. The evaluation time is explicit, and team lookups include the game season.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from hype_score.database import get_engine, load_games, load_team_ratings_for_games, load_pregame_odds, load_odds_snapshots

## Connect to Database

In [ ]:
evaluated_at = pd.Timestamp.now(tz="UTC")
engine = get_engine()
print(f"Connected to: {engine.url}")

## Games Overview

In [ ]:
games = load_games(engine)
print(f"Total games: {len(games):,}")
games.head()

In [ ]:
games_summary = games.groupby(['sport', 'status']).size().unstack(fill_value=0)
games_summary

In [ ]:
games['date'] = pd.to_datetime(games['scheduled_start']).dt.date
games_by_date = games.groupby(['date', 'sport']).size().reset_index(name='count')

fig = px.line(games_by_date, x='date', y='count', color='sport', title='Games Over Time')
fig.show()

## Team Ratings

In [ ]:
ratings = load_team_ratings_for_games(engine, games, evaluated_at=evaluated_at)
print(f"Total team ratings: {len(ratings):,}")
ratings.head()

In [ ]:
ratings.groupby(['sport', 'season']).agg({
    'team_abbreviation': 'nunique',
    'elo_rating': ['mean', 'min', 'max'],
    'bpi_rating': ['mean', 'min', 'max'],
    'playoff_probability': ['mean', 'min', 'max']
})

In [ ]:
for (sport, season), sport_ratings in ratings.groupby(['sport', 'season']):
    
    rating_col = 'elo_rating' if sport_ratings['elo_rating'].notna().any() else 'bpi_rating'
    if sport_ratings[rating_col].notna().any():
        fig = px.histogram(sport_ratings, x=rating_col, title=f'{sport.upper()} {season} {rating_col} Distribution')
        fig.show()

## Pregame Odds

In [ ]:
pregame_odds = load_pregame_odds(engine)
print(f"Games with pregame odds: {len(pregame_odds):,}")
pregame_odds.head()

In [ ]:
fig = px.histogram(
    pregame_odds, 
    x='home_win_probability', 
    nbins=50,
    title='Home Win Probability Distribution (Pregame)'
)
fig.add_vline(x=0.5, line_dash='dash', line_color='red', annotation_text='50/50')
fig.show()

## Odds Snapshots (Live Data)

In [ ]:
completed_games = games[games['status'] == 'final'].head(5)

for game_id in completed_games['id']:
    snapshots = load_odds_snapshots(engine, game_id=game_id)
    if len(snapshots) > 0:
        print(f"Game {game_id}: {len(snapshots)} snapshots")
        
        fig = px.line(
            snapshots, 
            x='timestamp', 
            y='home_win_probability',
            title=f'Win Probability Over Time - Game {game_id}'
        )
        fig.add_hline(y=0.5, line_dash='dash', line_color='gray')
        fig.update_yaxes(range=[0, 1])
        fig.show()
        break   

## Data Quality Summary

In [ ]:
games_with_odds = games.merge(pregame_odds[['game_id']], left_on='id', right_on='game_id', how='inner')

print(f"Games with pregame odds: {len(games_with_odds):,} / {len(games):,} ({100*len(games_with_odds)/len(games):.1f}%)")

coverage = games_with_odds.groupby('sport').size() / games.groupby('sport').size() * 100
print("\nOdds coverage by sport:")
for sport, pct in coverage.items():
    print(f"  {sport}: {pct:.1f}%")